In [1]:
import numpy as np
import pandas as pd

Set display options to inspect all columns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

Load the dataset

In [3]:
data_path = "../port_harcourt_air_quality_dataset/air_quality_historical.csv"
df_raw = pd.read_csv(data_path)
print(f"Dataset successfully loaded! Total rows: {df_raw.shape[0]}, Total columns: {df_raw.shape[1]}")
df_raw.head()

Dataset successfully loaded! Total rows: 1298, Total columns: 12


,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,30.969565,20.339130,288.608696,5.482609,3.786957,57.347826,0.361304,0.000,1.656522,NaN,NaN
4,2022-08-05,33.520833,22.720833,345.458333,5.912500,4.770833,62.500000,0.346250,0.375,1.375000,64.826087,38.434783


Inspect Column Headers

In [ ]:
#Inspect Column Headers for data consistency
df_raw.columns

Index(['date', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi'], dtype='str')

In [16]:
#Inspect data types and check for missing values
print("====Data Info===")
df_raw.info()


#Missing values audit
print("=====Missing Values Audit =======")
missing_summary = pd.DataFrame({
    "Missing Count": df_raw.isnull().sum(),
    "Missing Percentage (%)": (df_raw.isnull().sum() / len(df_raw)) * 100
})

missing_summary.sort_values(by="Missing Count", ascending=False)

====Data Info===
<class 'pandas.DataFrame'>
RangeIndex: 1298 entries, 0 to 1297
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1298 non-null   str    
 1   pm10                   1295 non-null   float64
 2   pm2_5                  1295 non-null   float64
 3   carbon_monoxide        1295 non-null   float64
 4   nitrogen_dioxide       1295 non-null   float64
 5   sulphur_dioxide        1295 non-null   float64
 6   ozone                  1295 non-null   float64
 7   aerosol_optical_depth  1295 non-null   float64
 8   dust                   1295 non-null   float64
 9   uv_index               1295 non-null   float64
 10  us_aqi                 1294 non-null   float64
 11  european_aqi           1294 non-null   float64
dtypes: float64(11), str(1)
memory usage: 134.5 KB
=====Missing Values Audit =======


,Missing Count,Missing Percentage (%)
european_aqi,4,0.308166
us_aqi,4,0.308166
pm10,3,0.231125
pm2_5,3,0.231125
aerosol_optical_depth,3,0.231125
carbon_monoxide,3,0.231125
nitrogen_dioxide,3,0.231125
sulphur_dioxide,3,0.231125
uv_index,3,0.231125
ozone,3,0.231125


Handling the missing data

In [17]:
# Display only the rows that contain at least one missing value
df_raw[df_raw.isnull().any(axis=1)]

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,30.969565,20.33913,288.608696,5.482609,3.786957,57.347826,0.361304,0.0,1.656522,NaN,NaN


The first 3 rows (2022-08-01, 2022-08-02, 2022-08-03): Completely empty across all pollutants. This is an initialization artifact—the satellite/sensor feed only began recording on August 4, 2022.
The 4th row (2022-08-04): The pollutants (pm2_5, pm10, co, etc.) were captured, but the calculated indices (us_aqi, european_aqi) were missing on the first active day.

Recommended: In data engineering, when missingness is < 0.5% and localized at the startup boundary of a sensor stream, dropping them is the most statistically sound method. You retain 1,296 clean, real-world rows without introducing artificial bias into the ML model.

In [ ]:
# 1. Convert 'date' to datetime format for proper time-series indexing
df_clean = df_raw.copy()
df_clean['date'] = pd.to_datetime(df_clean['date'])
# 2. Drop the 4 incomplete initialization rows
df_clean = df_clean.dropna().reset_index(drop=True)

In [ ]:
print("====Clean Dataset Check===")
clean_check = pd.DataFrame({
    "dirty_data_count": len(df_raw),
    "clean_data_count": len(df_clean),
    "Total_rows_dropped": len(df_raw) - len(df_clean),
    "Percentage_rows_droppped": (len(df_clean) - len(df_raw) / len(df_raw)) * 100:.2f)
}) 